# 데이터 누수(Data Leakage) 체크

BERT 모델 성능이 예상보다 높게 나온 이유를 검증한다.

- Binary F1: 0.9869 (CV) / 0.9881 (test_final)
- Multi F1: 0.8854 (CV) / 0.8960 (test_final)

**체크 항목:**
1. work_pool ↔ test_final 제목/본문 중복 여부
2. work_pool 내부 중복 → K-Fold leakage 여부
3. 라벨 정보가 입력 텍스트에 유출됐는지

In [ ]:
import pandas as pd
import numpy as np

# work_pool 로딩
print('work_pool 로딩 중...')
df_train = pd.concat([
    pd.read_parquet('data/processed/work_pool_clickbait_auto.parquet'),
    pd.read_parquet('data/processed/work_pool_clickbait_direct.parquet'),
    pd.read_parquet('data/processed/work_pool_nonclickbait_auto.parquet'),
], ignore_index=True)

# test_final 로딩 (봉인 해제 — 누수 체크 목적)
print('test_final 로딩 중...')
df_test = pd.read_parquet('data/processed/test_final.parquet')

print(f'\nwork_pool: {len(df_train):,}건')
print(f'test_final: {len(df_test):,}건')
print(f'컬럼: {list(df_train.columns)}')

---
## 체크 1. work_pool ↔ test_final 중복

In [ ]:
# 1-1. 제목 기준 중복
title_overlap = df_train['title_clean'].isin(df_test['title_clean'])
n_title = title_overlap.sum()

# 1-2. 본문 기준 중복
content_overlap = df_train['content_clean'].isin(df_test['content_clean'])
n_content = content_overlap.sum()

# 1-3. 제목+본문 모두 중복
df_train['_key'] = df_train['title_clean'] + '|||' + df_train['content_clean']
df_test['_key']  = df_test['title_clean']  + '|||' + df_test['content_clean']
both_overlap = df_train['_key'].isin(df_test['_key'])
n_both = both_overlap.sum()

print('=' * 50)
print('[ 체크 1 ] work_pool ↔ test_final 중복')
print('=' * 50)
print(f'  제목만 중복:       {n_title:,}건  ({n_title/len(df_train)*100:.3f}%)')
print(f'  본문만 중복:       {n_content:,}건  ({n_content/len(df_train)*100:.3f}%)')
print(f'  제목+본문 모두 중복: {n_both:,}건  ({n_both/len(df_train)*100:.3f}%)')
print()

if n_both == 0:
    print('✅ train ↔ test 완전 중복 없음 — 이 경로의 누수 없음')
else:
    print(f'❌ 완전 중복 {n_both}건 발견 — 누수 가능성 있음!')
    print('\n중복 샘플:')
    print(df_train[both_overlap][['newsID', 'title_clean', 'binary_label']].head(5))

In [ ]:
# 제목만 중복인 경우 — 라벨이 같은지 다른지 확인
if n_title > 0:
    print(f'[ 제목 중복 {n_title}건 상세 확인 ]')
    dup_titles = df_train.loc[title_overlap, ['title_clean', 'binary_label']]
    
    # test_final과 라벨 비교
    merged = dup_titles.merge(
        df_test[['title_clean', 'binary_label']].rename(columns={'binary_label': 'test_label'}),
        on='title_clean'
    )
    label_mismatch = (merged['binary_label'] != merged['test_label']).sum()
    print(f'  라벨 불일치: {label_mismatch}건 (같은 제목인데 라벨이 다름 → 심각한 문제)')
    print(f'  라벨 일치: {len(merged) - label_mismatch}건')
    print()
    print('샘플:')
    print(merged.head(5))
else:
    print('제목 중복 없음 ✅')

---
## 체크 2. work_pool 내부 중복 → K-Fold leakage

In [ ]:
# 2-1. 제목+본문 내부 중복
before = len(df_train)
df_dedup = df_train.drop_duplicates(subset=['title_clean', 'content_clean'])
after = len(df_dedup)
n_internal_dup = before - after

print('=' * 50)
print('[ 체크 2 ] work_pool 내부 중복')
print('=' * 50)
print(f'  중복 제거 전: {before:,}건')
print(f'  중복 제거 후: {after:,}건')
print(f'  제거된 중복: {n_internal_dup}건')
print()

if n_internal_dup <= 251:
    print(f'✅ EDA에서 발견한 251건과 일치 ({n_internal_dup}건) — 예상 범위 내')
    print('   bert_modeling.md에서 deduplication 처리했다고 명시되어 있음')
else:
    print(f'⚠️ 예상(251건)보다 많은 {n_internal_dup}건 중복 발견')

# 2-2. 중복 케이스에서 라벨이 다른 경우 확인 (더 심각한 문제)
dup_mask = df_train.duplicated(subset=['title_clean', 'content_clean'], keep=False)
dup_df = df_train[dup_mask].sort_values(['title_clean', 'content_clean'])

label_conflict = (
    dup_df.groupby(['title_clean', 'content_clean'])['binary_label']
    .nunique()
    > 1
).sum()
print(f'\n  중복 케이스 중 라벨 충돌(같은 기사, 다른 라벨): {label_conflict}건')
if label_conflict == 0:
    print('  ✅ 라벨 충돌 없음')
else:
    print('  ❌ 라벨 충돌 발견 — 심각한 데이터 품질 문제!')

---
## 체크 3. 라벨 정보 유출 여부

In [ ]:
print('=' * 50)
print('[ 체크 3 ] 라벨 정보 유출 여부')
print('=' * 50)

# 낚시성/정상 구분 키워드가 제목에 직접 노출됐는지
leakage_keywords = ['낚시', 'clickbait', '낚시성', '정상기사', 'nonclickbait', 'label=']

for kw in leakage_keywords:
    n = df_train['title_clean'].str.contains(kw, case=False, na=False).sum()
    if n > 0:
        print(f'  ❌ "{kw}" 포함: {n}건')
    else:
        print(f'  ✅ "{kw}" 포함 없음')

# newsID 패턴 확인 (클래스 정보가 ID에 포함됐는지)
print()
print('[ newsID 패턴 샘플 ]')
print('  낚시성:', df_train[df_train['binary_label']==1]['newsID'].head(3).tolist())
print('  정상:  ', df_train[df_train['binary_label']==0]['newsID'].head(3).tolist())

---
## 체크 4. 데이터 자체 난이도 분석 (높은 정확도가 당연한지 확인)

In [ ]:
print('=' * 50)
print('[ 체크 4 ] 데이터 난이도 분석')
print('=' * 50)

# 낚시성 vs 정상 기사의 제목 길이 차이
cb  = df_train[df_train['binary_label'] == 1]['title_clean']
ncb = df_train[df_train['binary_label'] == 0]['title_clean']

print(f'\n제목 길이 (평균):')
print(f'  낚시성: {cb.str.len().mean():.1f}자')
print(f'  정상:   {ncb.str.len().mean():.1f}자')

# 구두점 차이 (이미 EDA에서 확인됨)
q_cb  = cb.str.contains(r'\?').mean()
q_ncb = ncb.str.contains(r'\?').mean()
e_cb  = cb.str.contains(r'\.\.\.').mean()
e_ncb = ncb.str.contains(r'\.\.\.').mean()

print(f'\n구두점 비율:')
print(f'  ? 포함 — 낚시: {q_cb:.3f} / 정상: {q_ncb:.3f} → {q_cb/q_ncb:.1f}배')
print(f'  ... 포함 — 낚시: {e_cb:.3f} / 정상: {e_ncb:.3f} → {e_cb/e_ncb:.1f}배')

# 낚시성 기사 제목 샘플 (패턴 육안 확인)
print(f'\n낚시성 기사 제목 샘플 10건:')
for t in df_train[df_train['binary_label']==1]['title_clean'].sample(10, random_state=42).tolist():
    print(f'  [{t}]')

print(f'\n정상 기사 제목 샘플 10건:')
for t in df_train[df_train['binary_label']==0]['title_clean'].sample(10, random_state=42).tolist():
    print(f'  [{t}]')

---
## 최종 판정

In [ ]:
print('=' * 60)
print('[ 최종 누수 판정 ]')
print('=' * 60)
print(f'  체크 1 — train↔test 완전 중복: {n_both}건')
print(f'  체크 2 — work_pool 내부 중복:  {n_internal_dup}건 (예상 251건)')
print(f'  체크 3 — 라벨 키워드 유출:     위 출력 결과 확인')
print()

if n_both == 0 and n_internal_dup <= 260:
    print('종합 판정: ✅ 데이터 누수 없음')
    print()
    print('높은 정확도(0.9869)의 가능한 이유:')
    print('  1. 낚시성 기사 제목의 표면적 패턴이 명확함 (?, ..., 선정적 단어)')
    print('  2. KLUE-RoBERTa가 한국어 뉴스로 사전학습되어 도메인 친화적')
    print('  3. 29만 건 대규모 데이터로 fine-tuning → 충분한 학습')
    print('  4. Binary 분류는 50:50 균형 → 모델이 편향 없이 학습')
else:
    print('종합 판정: ❌ 누수 의심 — 위 결과 팀에 공유 필요')